# Scopus and OpenAlex Bronze-to-Silver evidence

This notebook reads the same mounted Bronze and Silver directories used by the scheduled container. Run it after an ingestion/processing cycle to inspect a live snapshot. It is intentionally read-only: it does not call either API and never accesses credentials.

## Evidence checklist

The cells below show source-level ingestion checkpoints, per-batch Spark processing results, Silver Parquet counts/schema/samples, DOI-deduplicated analysis counts, and rejection reasons. Empty zones are reported clearly, so the notebook is also safe to run before the first scheduled batch.

In [ ]:
from pathlib import Path
import json
import sys

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import ArrayType, DoubleType, LongType, StringType, StructField, StructType

BRONZE = Path("/data/bronze")
SILVER = Path("/data/silver")
PROJECT_ROOT = Path("/app")
if PROJECT_ROOT.exists() and str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
spark = (
    SparkSession.builder.master("local[*]")
    .appName("openalex-pipeline-evidence")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)


In [ ]:
# Batch metadata is the audit index for ingestion and processing state.
metadata_paths = sorted((BRONZE / "batch_metadata").glob("*.json"))
metadata_rows = []
for path in metadata_paths:
    item = json.loads(path.read_text(encoding="utf-8"))
    metadata_rows.append({
        "source": item.get("source", "openalex"),
        "batch_id": item.get("batch_id"),
        "mode": item.get("mode") or "unknown",
        "status": item.get("status") or "unknown",
        "record_count": int(item.get("record_count", 0) or 0),
        "page_count": int(item.get("page_count", 0) or 0),
        "created_at": str(item.get("created_at") or "unknown"),
        "completed_at": str(item.get("completed_at") or "not completed"),
    })
if metadata_rows:
    batch_metadata_df = spark.createDataFrame(metadata_rows)
    print(f"Batch metadata rows: {batch_metadata_df.count()}")
    batch_metadata_df.printSchema()
    display(batch_metadata_df.orderBy("source", "created_at"))
else:
    print("No batch metadata has been written yet.")


In [ ]:
# The checkpoint metadata gives a source-separated, resumable ingestion snapshot.
bronze_source_rows = (
    batch_metadata_df.groupBy("source")
    .agg(
        F.sum("record_count").alias("checkpointed_records"),
        F.sum("page_count").alias("checkpointed_pages"),
        F.countDistinct("batch_id").alias("batches"),
    )
    .orderBy("source")
) if metadata_rows else None
if bronze_source_rows is not None:
    display(bronze_source_rows)
    print(f"Total checkpointed Bronze records: {sum(int(row.get('record_count', 0) or 0) for row in metadata_rows):,}")

# Count actual raw page files independently from auxiliary Spark files.
raw_page_counts = []
for source in ("openalex", "scopus"):
    paths = sorted((BRONZE / source).glob("*/page-*.json"))
    raw_page_counts.append({"source": source, "raw_page_files": len(paths)})
if raw_page_counts:
    display(spark.createDataFrame(raw_page_counts).orderBy("source"))
else:
    print("No raw Bronze pages have been written yet.")


In [ ]:
# Processing state records valid/rejected rows and write duration per batch.
processing_rows = []
for item in metadata_rows:
    batch_id = item.get("batch_id")
    state_path = SILVER / "_processing" / f"{batch_id}.json"
    state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}
    batch_path = SILVER / "works" / str(batch_id)
    parquet_files = list(batch_path.glob("part-*.parquet")) if batch_path.exists() else []
    processing_rows.append({
        "source": item.get("source", "openalex"),
        "batch_id": batch_id,
        "ingestion_status": item.get("status"),
        "processing_status": state.get("status", "not_processed"),
        "valid_rows": state.get("valid_rows"),
        "rejected_rows": state.get("rejected_rows"),
        "write_seconds": state.get("write_seconds"),
        "silver_parquet_files": len(parquet_files),
        "silver_parquet_bytes": sum(path.stat().st_size for path in parquet_files),
    })
if processing_rows:
    processing_schema = StructType([
        StructField("source", StringType(), False),
        StructField("batch_id", StringType(), False),
        StructField("ingestion_status", StringType(), True),
        StructField("processing_status", StringType(), False),
        StructField("valid_rows", LongType(), True),
        StructField("rejected_rows", LongType(), True),
        StructField("write_seconds", DoubleType(), True),
        StructField("silver_parquet_files", LongType(), False),
        StructField("silver_parquet_bytes", LongType(), False),
    ])
    processing_df = spark.createDataFrame(processing_rows, schema=processing_schema)
    print(f"Batches with completed processing: {sum(row['processing_status'] == 'completed' for row in processing_rows)}")
    display(processing_df.orderBy("source", "batch_id"))
else:
    print("No ingestion batches are available to process yet.")


In [ ]:
# Resolve the published compaction manifest, if one exists, to avoid reading
# both original and compacted copies of the same Silver batches.
from src.compaction import active_silver_paths
silver_paths = active_silver_paths(SILVER)
if silver_paths:
    silver_df = spark.read.option("mergeSchema", "true").parquet(
        *(str(path) for path in silver_paths)
    )
    # Older Silver snapshots predate cross-source provenance columns.
    if "source" not in silver_df.columns:
        silver_df = silver_df.withColumn("source", F.lit("openalex"))
    else:
        silver_df = silver_df.withColumn("source", F.coalesce(F.col("source"), F.lit("openalex")))
    if "source_id" not in silver_df.columns and "openalex_id" in silver_df.columns:
        silver_df = silver_df.withColumn("source_id", F.col("openalex_id"))
    elif "source_id" in silver_df.columns and "openalex_id" in silver_df.columns:
        silver_df = silver_df.withColumn("source_id", F.coalesce(F.col("source_id"), F.col("openalex_id")))
    if "normalized_doi" not in silver_df.columns and "doi" in silver_df.columns:
        silver_df = silver_df.withColumn("normalized_doi", F.regexp_replace(
            F.lower(F.trim(F.col("doi"))),
            r"^(https?://(dx\.)?doi\.org/|doi:)",
            "",
        ))
    if "normalized_doi" not in silver_df.columns:
        silver_df = silver_df.withColumn("normalized_doi", F.lit(None).cast("string"))
    print(f"Silver rows read: {silver_df.count():,}")
    silver_df.printSchema()
    display(silver_df.groupBy("source").count().orderBy("source"))
    sample_columns = [name for name in ("source", "source_id", "batch_id", "title", "publication_year", "doi", "normalized_doi") if name in silver_df.columns]
    display(silver_df.select(*sample_columns).orderBy("source", "source_id").limit(10))
    from src.spark_pipeline import unique_publications
    unique_df = unique_publications(silver_df)
    print(f"DOI-deduplicated analysis-view rows: {unique_df.count():,}")
    display(unique_df.groupBy("source").count().orderBy("source"))
else:
    print("No published Silver Parquet batches have been written yet.")


In [ ]:
rejected_paths = sorted((BRONZE / "rejected").glob("*/part-*.json"))
if rejected_paths:
    rejected_df = spark.read.json([str(path) for path in rejected_paths])
    print(f"Rejected rows: {rejected_df.count()}")
    rejected_df.printSchema()
    display(
        rejected_df.groupBy("rejection_reason")
        .count()
        .orderBy(F.desc("count"))
    )
else:
    print("No rejected rows have been written yet.")
